# Pydantic 数据校验工具

这个 notebook 演示如何用 Pydantic v2 编写可复用的数据校验工具，覆盖模型定义、字段约束、计算字段、批量校验、JSON 序列化、JSON Schema 和错误整理。代码中的类与函数保留 Google 风格中文 docstring，便于直接复制到项目模块中使用。

In [ ]:
"""Pydantic 数据校验工具示例。"""

from collections.abc import Mapping, Sequence
from decimal import Decimal
from typing import Any, Literal

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    TypeAdapter,
    ValidationError,
    computed_field,
    field_validator,
)

OrderStatus = Literal["pending", "paid", "cancelled"]


class OrderItem(BaseModel):
    """订单明细模型。

    Attributes:
        sku_id: 商品 SKU 编号。
        quantity: 购买数量，必须大于 0。
        unit_price: 商品单价，最多保留两位小数。
        line_total: 当前明细的金额小计。
    """

    model_config = ConfigDict(frozen=True, str_strip_whitespace=True)

    sku_id: str = Field(min_length=1, description="商品 SKU 编号")
    quantity: int = Field(gt=0, description="购买数量")
    unit_price: Decimal = Field(
        gt=Decimal("0"),
        max_digits=12,
        decimal_places=2,
        description="商品单价",
    )

    @field_validator("unit_price", mode="before")
    @classmethod
    def normalize_unit_price(cls, value: Any) -> Any:
        """规范化商品单价。

        Args:
            value: 用户输入的原始单价值。

        Returns:
            适合 Pydantic 继续解析的单价值。浮点数会先转成字符串，
            以减少二进制浮点表示带来的精度噪音。
        """
        if isinstance(value, float):
            return str(value)
        return value

    @computed_field
    @property
    def line_total(self) -> Decimal:
        """计算当前明细的小计金额。

        Returns:
            购买数量乘以商品单价后的金额。
        """
        return self.unit_price * self.quantity


class Order(BaseModel):
    """订单模型。

    Attributes:
        order_id: 订单编号。
        user_id: 用户编号。
        status: 订单状态，支持 pending、paid 和 cancelled。
        items: 订单明细列表，至少包含一条明细。
        total_amount: 订单总金额。
    """

    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)

    order_id: str = Field(min_length=1, description="订单编号")
    user_id: str = Field(min_length=1, description="用户编号")
    status: OrderStatus = Field(default="pending", description="订单状态")
    items: list[OrderItem] = Field(min_length=1, description="订单明细")

    @computed_field
    @property
    def total_amount(self) -> Decimal:
        """计算订单总金额。

        Returns:
            所有订单明细小计金额的总和。
        """
        return sum(item.line_total for item in self.items)


_ORDER_LIST_ADAPTER = TypeAdapter(list[Order])


def parse_order(payload: Mapping[str, Any] | str | bytes | bytearray) -> Order:
    """解析并校验单个订单。

    Args:
        payload: 订单字典、JSON 字符串或 JSON 字节串。

    Returns:
        校验通过后的订单模型。

    Raises:
        ValidationError: 当字段缺失、类型错误或业务约束不满足时抛出。
    """
    if isinstance(payload, str | bytes | bytearray):
        return Order.model_validate_json(payload)
    return Order.model_validate(payload)


def parse_orders(payloads: Sequence[Mapping[str, Any]] | str | bytes | bytearray) -> list[Order]:
    """批量解析并校验订单。

    Args:
        payloads: 订单字典序列、JSON 字符串或 JSON 字节串。

    Returns:
        校验通过后的订单模型列表。

    Raises:
        ValidationError: 当任意订单字段缺失、类型错误或业务约束不满足时抛出。
    """
    if isinstance(payloads, str | bytes | bytearray):
        return _ORDER_LIST_ADAPTER.validate_json(payloads)
    return _ORDER_LIST_ADAPTER.validate_python(payloads)


def dump_order(order: Order) -> dict[str, Any]:
    """把订单模型转换为适合 JSON 输出的字典。

    Args:
        order: 已校验的订单模型。

    Returns:
        JSON 友好的订单字典。Decimal 字段会转换成字符串，计算字段也会包含在结果中。
    """
    return order.model_dump(mode="json")


def order_schema() -> dict[str, Any]:
    """生成订单模型的 JSON Schema。

    Returns:
        可用于接口文档或前端表单生成的 JSON Schema 字典。
    """
    return Order.model_json_schema()


def summarize_validation_error(error: ValidationError) -> list[str]:
    """整理 Pydantic 校验错误。

    Args:
        error: Pydantic 抛出的校验异常。

    Returns:
        更适合日志或接口响应展示的错误消息列表。
    """
    messages: list[str] = []
    for item in error.errors():
        location = ".".join(str(part) for part in item["loc"])
        message = item["msg"]
        messages.append(f"{location}: {message}" if location else message)
    return messages


## 解析单个订单

Pydantic 会自动清理字符串首尾空白、校验字段约束，并生成 `line_total` 与 `total_amount` 这类计算字段。

In [ ]:
sample_order_payload = {
    "order_id": " ORD-001 ",
    "user_id": " USER-001 ",
    "status": "paid",
    "items": [
        {"sku_id": " SKU-1 ", "quantity": 2, "unit_price": "12.50"},
        {"sku_id": "SKU-2", "quantity": 1, "unit_price": 3.5},
    ],
}

order = parse_order(sample_order_payload)
dumped_order = dump_order(order)

assert order.order_id == "ORD-001"
assert order.items[0].sku_id == "SKU-1"
assert order.total_amount == Decimal("28.50")
assert dumped_order["total_amount"] == "28.50"

dumped_order


## 批量校验和生成 Schema

`TypeAdapter` 适合处理列表、联合类型等非 `BaseModel` 根类型；`model_json_schema()` 可以把模型约束转换成接口文档或表单系统可消费的结构。

In [ ]:
import json

raw_orders = json.dumps(
    [
        {
            "order_id": "ORD-002",
            "user_id": "USER-002",
            "items": [{"sku_id": "SKU-8", "quantity": 2, "unit_price": "19.90"}],
        }
    ],
    ensure_ascii=False,
)

orders = parse_orders(raw_orders)
schema = order_schema()

assert len(orders) == 1
assert orders[0].status == "pending"
assert orders[0].total_amount == Decimal("39.80")
assert schema["title"] == "Order"

schema["properties"]["items"]


## 整理校验错误

实际项目通常不会直接把 `ValidationError` 原样返回给调用方，可以先整理出稳定的字段路径和错误消息。

In [ ]:
try:
    parse_order(
        {
            "order_id": "ORD-003",
            "user_id": "USER-003",
            "items": [{"sku_id": "", "quantity": 0, "unit_price": "0"}],
        }
    )
except ValidationError as error:
    validation_messages = summarize_validation_error(error)

assert any(message.startswith("items.0.sku_id:") for message in validation_messages)
assert any(message.startswith("items.0.quantity:") for message in validation_messages)
assert any(message.startswith("items.0.unit_price:") for message in validation_messages)

validation_messages
